# 📊 01 — Exploratory Data Analysis

Explore the chest X-ray dataset: class distribution, image statistics,
sample visualization, and pixel intensity analysis.

**Dataset**: Kaggle Chest X-Ray Images (Pneumonia) — re-split 80/10/10

In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

DATA_ROOT = Path('../data/processed')
SPLITS = ['train', 'val', 'test']
CLASSES = ['NORMAL', 'PNEUMONIA']

print(f'Data root: {DATA_ROOT.resolve()}')
print(f'Exists: {DATA_ROOT.exists()}')

## 1. Class Distribution

In [ ]:
# Count images per split per class
counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        cls_dir = DATA_ROOT / split / cls
        n = len(list(cls_dir.glob('*'))) if cls_dir.exists() else 0
        counts[split][cls] = n

# Display table
print(f"{'Split':<8} {'NORMAL':>8} {'PNEUMONIA':>10} {'Total':>8} {'% Pneumonia':>12}")
print('─' * 50)
for split in SPLITS:
    n = counts[split]['NORMAL']
    p = counts[split]['PNEUMONIA']
    total = n + p
    pct = p / total * 100 if total > 0 else 0
    print(f'{split:<8} {n:>8} {p:>10} {total:>8} {pct:>11.1f}%')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, split in enumerate(SPLITS):
    values = [counts[split][cls] for cls in CLASSES]
    colors = ['#4CAF50', '#F44336']
    axes[idx].bar(CLASSES, values, color=colors, edgecolor='white', linewidth=1.5)
    axes[idx].set_title(f'{split.upper()} ({sum(values)} images)', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Count')
    for j, v in enumerate(values):
        axes[idx].text(j, v + 20, str(v), ha='center', fontweight='bold')

plt.suptitle('Class Distribution Across Splits', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/class_distribution.png', dpi=150, bbox_inches='tight', transparent=True)
plt.show()

## 2. Sample Images

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for row, cls in enumerate(CLASSES):
    cls_dir = DATA_ROOT / 'train' / cls
    images = sorted(cls_dir.glob('*'))[:5]
    for col, img_path in enumerate(images):
        img = Image.open(img_path).convert('L')  # grayscale
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=13, fontweight='bold', rotation=0, labelpad=80)

plt.suptitle('Sample Chest X-Rays', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../evaluation/sample_images.png', dpi=150, bbox_inches='tight', transparent=True)
plt.show()

## 3. Image Size Distribution

In [ ]:
# Analyze image dimensions
widths, heights, labels = [], [], []

for cls in CLASSES:
    cls_dir = DATA_ROOT / 'train' / cls
    for img_path in list(cls_dir.glob('*'))[:200]:  # sample 200 per class
        try:
            img = Image.open(img_path)
            w, h = img.size
            widths.append(w)
            heights.append(h)
            labels.append(cls)
        except Exception:
            pass

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax1.set_title('Width Distribution')
ax1.set_xlabel('Pixels')
ax2.hist(heights, bins=30, color='coral', edgecolor='white', alpha=0.8)
ax2.set_title('Height Distribution')
ax2.set_xlabel('Pixels')
plt.tight_layout()
plt.show()

## 4. Pixel Intensity Analysis

In [ ]:
# Compare pixel intensity distributions between classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, cls in enumerate(CLASSES):
    cls_dir = DATA_ROOT / 'train' / cls
    intensities = []
    for img_path in list(cls_dir.glob('*'))[:100]:
        img = np.array(Image.open(img_path).convert('L').resize((64, 64)))
        intensities.append(img.flatten())
    
    all_pixels = np.concatenate(intensities)
    axes[idx].hist(all_pixels, bins=50, density=True, color=['#4CAF50', '#F44336'][idx],
                   edgecolor='white', alpha=0.8)
    axes[idx].set_title(f'{cls} — Pixel Intensity', fontweight='bold')
    axes[idx].set_xlabel('Pixel Value (0-255)')
    axes[idx].set_ylabel('Density')
    axes[idx].axvline(all_pixels.mean(), color='black', linestyle='--', label=f'Mean: {all_pixels.mean():.0f}')
    axes[idx].legend()

plt.suptitle('Pixel Intensity Distribution by Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../evaluation/pixel_intensity.png', dpi=150, bbox_inches='tight', transparent=True)
plt.show()

## 5. Key Observations

- **Class imbalance**: ~73% Pneumonia across all splits — handled by weighted loss + weighted sampler
- **Image size variability**: Different equipment produces different resolutions — resizing to 224×224 normalizes this
- **Pixel intensity**: Pneumonia cases tend to have slightly different intensity distributions due to lung opacities
- **Data quality**: Some images have annotations/markers that could introduce noise